# Lesson 4.6 — Multi-Instance & Multi-Pass Review Architectures

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why self-review is weaker than an independent review instance
- Use a second independent Claude instance to review generated code
- Split large reviews into per-file passes plus a cross-file integration pass
- Run verification passes with self-reported confidence for calibrated routing

**You will produce:** An independent-reviewer pipeline plus a per-file + integration pass splitter.

## Concepts

**Self-review is limited:** a model retains reasoning context from generation, so it is less likely to question its own decisions in the same session. **Independent review instances** (without the generator's reasoning context) catch subtle issues better than self-review instructions or extended thinking.

**Multi-pass review:** split large multi-file reviews into **per-file local passes** plus a separate **cross-file integration pass** to avoid attention dilution and contradictory findings. Sample Question 12: a 14-file single pass gave inconsistent, contradictory feedback - the fix is focused passes, NOT a bigger context window and NOT consensus voting (which suppresses real bugs).

Run **verification passes** where the model self-reports **confidence** alongside each finding to enable calibrated review routing (see Lesson 5.5).

> **Reference:** [Anthropic - reduce hallucinations / verify](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Independent reviewer (fresh context)

The reviewer never sees the generator's reasoning - only the code.

In [ ]:
def generate_code(spec):
    return "def add(a,b):\n    return a-b  # bug: should be a+b"

def independent_review(code):
    # New instance: ONLY the code, no generation rationale.
    prompt = f"Review this code for bugs. Report location/issue/severity.\n\n{code}"
    if HAVE_API:
        resp = client.messages.create(model=MODEL_SONNET, max_tokens=400,
            messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in resp.content if b.type == 'text')
    return "[MOCK] high: subtraction used where addition intended (add)"

code = generate_code("add two numbers")
print(independent_review(code))

## Code-Along: Step 2 — Per-file + integration passes

Local passes find local bugs; one integration pass checks cross-file data flow.

In [ ]:
def review_plan(files):
    plan = [{"pass": "local", "file": f} for f in files]
    plan.append({"pass": "integration", "scope": "cross-file data flow"})
    return plan

for step in review_plan([f"module_{i}.py" for i in range(1, 4)]):
    print(step)

## Code-Along: Step 3 — Confidence-annotated verification pass

Attaching confidence enables calibrated routing to human review later.

In [ ]:
def verify_with_confidence(findings):
    # Model would self-report; we annotate for the routing demo.
    for f in findings:
        f.setdefault("confidence", 0.8)
    return findings

print(verify_with_confidence([{"issue": "off-by-one"}]))

## Exercise

Reject the anti-pattern from Sample Question 12: implement `restructure_review(num_files)` that returns a per-file + integration plan for >5 files, and explicitly does NOT return a 'bigger context window' or 'consensus voting' strategy. Then route findings with confidence < 0.6 to human review.

In [ ]:
# TODO: implement restructure_review(num_files) -> focused passes
# TODO: assert the strategy is not 'bigger_context' or 'consensus_voting'
# TODO: route confidence < 0.6 findings to human review

## Key Takeaways

- Self-review is weaker; use an independent instance without the generator's context.
- Split large reviews into per-file passes + one cross-file integration pass.
- Bigger context windows and consensus voting do NOT fix attention dilution.
- Self-reported confidence per finding enables calibrated review routing.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.